# Ανίχνευση Εισβολών (Testing Phase)
Covers steps **4.1 → 4.5**.

| Step | Task |
|------|------|
| 4.1  | Load config, models, and test data (Benign + Attack) |
| 4.2  | Mix & sort traffic to create chronologically valid test sequence |
| 4.3  | Compute conditional entropy (cross-entropy) formula |
| 4.4  | Evaluate over time-based windows |
| 4.5  | Define alarm threshold and flag anomalies (Plotly interactive charts) |

## 4.1 – Βιβλιοθήκες & Imports
Εισαγωγή των απαραίτητων εργαλείων (Polars, Plotly, κλπ).

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

NOTEBOOK_DIR = Path(os.getcwd()).resolve()
PROJECT_ROOT = None

for candidate in [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents):
    if (candidate / "config.yaml").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None and Path("/content/drive/MyDrive").exists():
    # Fast paths to check instead of slow rglob
    candidate_paths = [
        Path("/content/drive/MyDrive/Entropy-Based Network Intrusion Detection using Markov Chains"),
        Path("/content/drive/MyDrive/IT_PROJECT"),
        Path("/content/drive/MyDrive/IT_Project"),
    ]
    for p in candidate_paths:
        if (p / "config.yaml").exists():
            PROJECT_ROOT = p
            break

if PROJECT_ROOT is None:
    PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name in ["2_Data_Pipeline", "3_Training", "4_Detection", "5_Evaluation"] else NOTEBOOK_DIR

os.chdir(PROJECT_ROOT)
print(f"PROJECT_ROOT verified: {PROJECT_ROOT}")
DATA_PATH = PROJECT_ROOT / "data"


In [ ]:
import polars as pl
import numpy as np
import yaml
import json
import os
import plotly.graph_objects as go
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timedelta


## 4.1 – Φόρτωση Ρυθμίσεων & Μοντέλων (Config & Models)
Φορτώνουμε το config.yaml και τα pretrained transition matrices από το Phase 3.

In [ ]:

with open(PROJECT_ROOT / "config.yaml", "r") as f:
    config = yaml.safe_load(f)

models_dir = PROJECT_ROOT / "3_Training" / "models"
with open(models_dir / "transition_matrices.json", "r") as f:
    transition_matrices = json.load(f)
    
with open(models_dir / "baselines.json", "r") as f:
    baselines = json.load(f)

K_ORDER = config["model"]["recommended_k"]
SMOOTHING = config["discretization"]["zero_prob_smoothing"]
THRESHOLD_FACTOR = config["detection"]["threshold_factor"]
EPOCH_DUR = config["detection"]["epoch_duration_seconds"]

print(f"Loaded config. Using k={K_ORDER}, threshold_factor={THRESHOLD_FACTOR}")

## 4.2 – Ανάμειξη & Ταξινόμηση Δεδομένων (Data Mixing)
Ενώνουμε το 20% των κανονικών δεδομένων (test) με τα attacks και τα ταξινομούμε χρονολογικά.

In [ ]:
# Load benign test set and attack set
df_benign_test = pl.read_parquet(DATA_PATH / "benign_test.parquet")
df_attack = pl.read_parquet(DATA_PATH / "attack.parquet")

# Mix data (the data is now treated as a pure chronological sequence of flows)
df_test = pl.concat([df_benign_test, df_attack])

print(f"Test Set Size: {df_test.height} rows")
print(f"Attack samples: {df_attack.height}")
print(f"Benign test samples: {df_benign_test.height}")


## 4.3 – Συνάρτηση Υπολογισμού Cross-Entropy
Υπολογισμός του Negative Log-Likelihood (έκπληξη/απόκλιση) για κάθε sequence βάσει του baseline matrix.

In [ ]:
def window_cross_entropy(sequence, baseline_tm, k=1, smoothing=1e-9):
    """
    Calculate the cross-entropy (negative log-likelihood) of a sequence 
    given the baseline transition matrix from training.
    """
    if len(sequence) <= k:
        return 0.0
        
    log_prob_sum = 0.0
    for i in range(len(sequence) - k):
        state = str(tuple(sequence[i:i+k]))
        nxt = str(sequence[i+k])
        
        # Get transition probability from baseline model, else use smoothing
        if state in baseline_tm and nxt in baseline_tm[state]:
            p = baseline_tm[state][nxt]
        else:
            p = smoothing
            
        log_prob_sum += -np.log2(p)
        
    return log_prob_sum / (len(sequence) - k)

## 4.4 – Κυλιόμενο Παράθυρο & Αξιολόγηση (Sliding Window Evaluation)
Σαρώνουμε τα δεδομένα σε blocks των 5 λεπτών και υπολογίζουμε το Cross-Entropy.

In [ ]:
window_size = config["detection"]["packet_window_size"]
overlap = config["detection"]["packet_window_overlap"]
stride = window_size - overlap 

# Raise the threshold to eliminate false positives
THRESHOLD_FACTOR = 1.5 

features = [
    "Flow_Duration_sym", "Out_Degree_sym", 
    "Tot_Fwd_Pkts_sym", "TotLen_Fwd_Pkts_sym", "TCP_Flags_Int_sym"
]

# Pre-load baselines and transition matrices for all 5 features
models, base_H, thresh, seqs = {}, {}, {}, {}
for f in features:
    key = f"{f}_k{K_ORDER}"
    models[f] = transition_matrices[key]
    base_H[f] = baselines[key]
    thresh[f] = base_H[f] * THRESHOLD_FACTOR
    seqs[f] = df_test[f].to_list()

label_full = df_test["Label"].to_list()

window_indices, window_attack_counts, final_alerts, max_normalized_deviations = [], [], [], []
feature_deviations = {f: [] for f in features}

# Slide over the data evaluating all 5 sensors at once
for i in range(0, len(label_full) - window_size + 1, stride):
    labels_window = label_full[i : i + window_size]
    attacks = sum(1 for lbl in labels_window if lbl != "Benign")
    
    is_alert = False
    max_norm_dev = 0.0
    
    for f in features:
        seq_window = seqs[f][i : i + window_size]
        ce = window_cross_entropy(seq_window, models[f], k=K_ORDER, smoothing=SMOOTHING)
        dev = abs(ce - base_H[f])
        
        # Normalize: > 1.0 means it crossed the threshold
        norm_dev = dev / thresh[f] if thresh[f] > 0 else 0
        feature_deviations[f].append(norm_dev)
        
        if norm_dev > 1.0:
            is_alert = True
        if norm_dev > max_norm_dev:
            max_norm_dev = norm_dev

    window_indices.append(i)
    window_attack_counts.append(attacks)
    final_alerts.append(1 if is_alert else 0)
    max_normalized_deviations.append(max_norm_dev)

print(f"Evaluated {len(window_indices)} sequential windows.")
print(f"Total Alerts: {sum(final_alerts)} / {len(final_alerts)} windows")


## 4.5 – Υπολογισμός Κατωφλίου & Οπτικοποίηση με Plotly (Thresholding & Visualization)
Συγκρίνουμε την απόκλιση με το επιτρεπτό όριο και σχεδιάζουμε το διαδραστικό γράφημα Plotly.

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
for i, f in enumerate(features):
    fig.add_trace(go.Scatter(
        x=window_indices,
        y=feature_deviations[f],
        mode='lines',
        name=f.replace('_sym', ''),
        line=dict(width=1, color=colors[i]),
        opacity=0.7
    ))

if window_indices:
    fig.add_trace(go.Scatter(
        x=[window_indices[0], window_indices[-1]],
        y=[1.0, 1.0],
        mode='lines',
        name='Universal Threshold (1.0)',
        line=dict(color='black', dash='dash', width=3)
    ))

attack_indices = [window_indices[i] for i, count in enumerate(window_attack_counts) if count > 0]
if attack_indices:
    fig.add_trace(go.Scatter(
        x=attack_indices,
        y=[1.5] * len(attack_indices),
        mode='markers',
        name='Attack Present',
        marker=dict(color='red', symbol='triangle-down', size=8)
    ))

fig.update_layout(
    title=f'Multi-Sensor Normalized Entropy Deviation (k={K_ORDER})',
    xaxis_title='Flow Sequence Index',
    yaxis_title='Normalized Deviation (Threshold = 1.0)',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()


In [ ]:
import polars as pl

y_true = [1 if c > 0 else 0 for c in window_attack_counts]

results_dir = PROJECT_ROOT / "5_Evaluation" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

results_df = pl.DataFrame({
    "Window_Index": window_indices,
    "Deviation": max_normalized_deviations,
    "y_true": y_true,
    "y_pred": final_alerts
})
results_df.write_csv(results_dir / "detection_results.csv")
print(f"Saved multi-sensor detection results to {results_dir}")
